In [1]:
import re
from pathlib import Path
import fitz
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import ollama
#import pdfplumber

In [2]:
def extract_chapters_pypdf(pdf_path: str):
    reader = PdfReader(pdf_path)
    outlines = reader.outline
    
    # 1. Pobieramy listę tytułów i stron startowych z zakładek PDF
    chapters_info = []
    for entry in outlines:
        if not isinstance(entry, list):  # Interesują nas główne rozdziały
            try:
                title = entry.get('/Title')
                page_num = reader.get_destination_page_number(entry)
                chapters_info.append({"title": title, "start_page": page_num})
            except:
                continue

    # 2. Wyliczamy zakresy stron (od - do) dla każdego rozdziału
    for i in range(len(chapters_info) - 1):
        chapters_info[i]["end_page"] = chapters_info[i+1]["start_page"]
    
    # Ostatni rozdział trwa do końca dokumentu
    if chapters_info:
        chapters_info[-1]["end_page"] = len(reader.pages)

    # 3. Ekstrakcja tekstu z podziałem na rozdziały
    structured_data = []
    for ch in chapters_info:
        chapter_text = ""
        for pg_idx in range(ch["start_page"], ch["end_page"]):
            page = reader.pages[pg_idx]
            chapter_text += page.extract_text() + "\n"
        
        structured_data.append({
            "title": ch["title"],
            "content": chapter_text
        })
        
    return structured_data

In [3]:
def clean_structured_text(chapter_data):
    cleaned_chapters = []
    
    for ch in chapter_data:
        raw_text = ch['content']
        
        text = re.sub(r"[ \t]+", " ", raw_text)
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        lines = [line.strip() for line in text.split("\n")]
        lines = [line for line in lines if line]
        clean_content = "\n".join(lines)
        
        cleaned_chapters.append({
            "title": ch['title'],
            "content": clean_content
        })
        
    return cleaned_chapters

In [4]:
def chunk_structured_text(cleaned_chapters, chunk_size=800, overlap=150):
    all_chunks = []
    all_metadatas = []

    for ch in cleaned_chapters:
        title = ch['title']
        content = ch['content']
        text_length = len(content)
        start = 0

        # Wykonujemy chunking wewnątrz konkretnego rozdziału
        while start < text_length:
            end = start + chunk_size
            chunk = content[start:end]
            
            all_chunks.append(chunk)
            
            # Tworzymy słownik metadanych dla każdego kawałka
            # Możesz tu dodać więcej info, np. nazwę pliku
            all_metadatas.append({
                "chapter": title,
                "source": "Robodrill_Manual"
            })
            
            # Przesuwamy okno o chunk_size minus overlap
            start += chunk_size - overlap

    return all_chunks, all_metadatas

### Model Embeddingowy

In [5]:
def load_embedding_model():
    return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [6]:
def create_vector_db(chunks, metadatas, model, persist_dir="robodrill_db"):
    # Inicjalizacja klienta i kolekcji
    client = chromadb.PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection("robodrill")

    # Generowanie embeddingów dla wszystkich kawałków tekstu
    embeddings = model.encode(chunks).tolist()
    
    # Generowanie unikalnych ID (pozostaje bez zmian)
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    # KLUCZOWA ZMIANA: Dodajemy argument 'metadatas'
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadatas  # <--- To łączy tekst z informacją o rozdziale
    )

    return collection

In [14]:
def main():
    # 1. Ścieżka do dokumentu
    pdf_path = r"Robodrill_Operation_And_Maitenance_Handbook_16i_160i_160is_18i_180i.pdf"

    print("Rozpoczynam przetwarzanie dokumentu...")

    # 2. Ekstrakcja strukturalna (podział na rozdziały na podstawie zakładek PDF)
    structured_data = extract_chapters_pypdf(pdf_path)
    print(f"Pomyślnie wyodrębniono {len(structured_data)} rozdziałów.")

    # 3. Czyszczenie tekstu wewnątrz rozdziałów
    # Przechodzimy przez każdy rozdział i usuwamy zbędne znaki
    cleaned_data = clean_structured_text(structured_data)

    # 4. Inteligentny chunking z generowaniem metadanych
    chunks, metadatas = chunk_structured_text(cleaned_data, chunk_size=1500, overlap=300)
    print(f"Utworzono {len(chunks)} fragmentów z przypisanymi metadanymi.")

    # 5. Inicjalizacja modelu embeddingowego
    model = load_embedding_model()

    # 6. Tworzenie bazy wektorowej
    create_vector_db(chunks, metadatas, model, persist_dir="robodrill_db")

    print("\nPipeline zakończony — baza wektorowa z metadanymi gotowa.")

In [15]:
if __name__ == "__main__":
    main()

Rozpoczynam przetwarzanie dokumentu...
Pomyślnie wyodrębniono 22 rozdziałów.
Utworzono 1184 fragmentów z przypisanymi metadanymi.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Pipeline zakończony — baza wektorowa z metadanymi gotowa.


In [16]:
def ask_the_manual(query):
    # Załadowanie bazy
    client = chromadb.PersistentClient(path="robodrill_db")
    collection = client.get_collection("robodrill")
    model = load_embedding_model()
    
    # Szukamy 3 najbardziej podobnych fragmentów
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=3)
    
    print(f"\nSZUKANE HASŁO: {query}")
    print("="*50)
    
    for i in range(len(results['documents'][0])):
        text = results['documents'][0][i]
        meta = results['metadatas'][0][i] # Tu siedzi informacja o rozdziale!
        
        print(f"WYNIK {i+1} | ROZDZIAŁ: {meta['chapter']}")
        print(f"FRAGMENT: {text[:800]}...") 
        print("-" * 30)

In [17]:
ask_the_manual("Machine Spindle Error on the program, macro allarm")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SZUKANE HASŁO: Machine Spindle Error on the program, macro allarm
WYNIK 1 | ROZDZIAŁ: 13. DIAGNOSIS INFORMATION
FRAGMENT: s output.
A43 : A prohibited T code is specified after M06.
A95 : M06 is specified while the Z –axis machine coordinate is positive.
A96 : The current tool number parameter (PRM7810) is set to 0.
A97 : M06 is specified in canned cycle mode. M06 is specified in a block
containing the command instructing reference position return.
M06 is specified in tool compensation mode.
A98 : After the power was turned on or after an emergency stop was
released, M06 was specified before the first reference position
return. While the tool was being changed, machine lock was
enabled for the Z–axis.
A99 : A pry alarm occurred while the tool was being changed.
#7 #6 #5 #4 #3 #2 #1 #0
531DGN 585 584 583 582 581 580 502
502 : Large spindle distribution (system error)
580 : Spindle servo alarm (excessive error ...
------------------------------
WYNIK 2 | ROZDZIAŁ: 8. ALARM LIST
FRAGMENT

In [18]:
ask_the_manual("ILLEGAL PROGRAM NUMBER")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SZUKANE HASŁO: ILLEGAL PROGRAM NUMBER
WYNIK 1 | ROZDZIAŁ: 8. ALARM LIST
FRAGMENT:  not found.
Or the program with specified program
number was not found in program number
search. Check the data.
072 TOO MANY PROGRAMS The number of programs to be stored ex-
ceeded 63 (basic), 125 (option), 200 (op-
tion), 400 (option) or 1000 (option). Delete
unnecessary programs and execute pro-
gram registration again.
8
8.1 Alarms Displayed on NC Screen
470
No. Contents Message
073 PROGRAM NUMBER
ALREADY IN USE
The commanded program number has al-
ready been used.
Change the program number or delete un-
necessary programs and execute program
registration again.
074 ILLEGAL PROGRAM
NUMBER
The program number is other than 1 to
9999. Modify the program number.
075 PROTECT An attempt was made to register a program
whose number was protected.
076 ADDRESS P NOT DEFINED Address P (program nu...
------------------------------
WYNIK 2 | ROZDZIAŁ: 8. ALARM LIST
FRAGMENT:  the program number.
075 PROTECT An at

In [ ]:
# 1. Połączenie z bazą (inicjalizacja klienta i kolekcji)
client = chromadb.PersistentClient(path="robodrill_db")
collection = client.get_collection("robodrill")

# 2. Sprawdzenie jakie rozdziały są w bazie (wyciągamy metadane)
all_data = collection.get(include=['metadatas'])
unique_chapters = set([m['chapter'] for m in all_data['metadatas']])

print("ROZDZIAŁY ZNALEZIONE W BAZIE:")
for ch in sorted(list(unique_chapters)):
    print(f"- {ch}")

print("\n" + "="*50 + "\n")

# 3. Podejrzenie treści z Rozdziału 8
print("PRÓBKA TEKSTU Z ROZDZIAŁU 8:")
res = collection.get(where={"chapter": "8. ALARM LIST"}, limit=3)

if not res['documents']:
    print("BŁĄD: Rozdział 8 jest pusty w bazie!")
else:
    for doc in res['documents']:
        print(f"--- Fragment ---\n{doc[:500]}...\n")

ROZDZIAŁY ZNALEZIONE W BAZIE:
- 1. SCREEN DISPLAY AND OPERATION
- 10. PMC
- 11. ETHERNET
- 12. POWER MATE CNC MANAGER (FANUC SERVO MOTOR AMPLIFIER β SERIES (I/O LINK OPTION))
- 13. DIAGNOSIS INFORMATION
- 14. HISTORY FUNCTION
- 15. WAVEFORM DIAGNOSTIC FUNCTION
- 16. DIGITAL SERVO
- 17. AC SPINDLE
- 18. MAINTENANCE INFORMATION
- 19. MAINTENANCE FUNCTION
- 2. OPERATION LIST
- 3. G CODE
- 4. PROGRAM FORMAT
- 5. CUSTOM MACRO
- 6. HARDWARE
- 7. PARAMETERS
- 8. ALARM LIST
- 9. SIGNAL LIST (X/Y, G/F)
- PREFACE
- SAFETY PRECAUTIONS
- TABLE OF CONTENTS


PRÓBKA TEKSTU Z ROZDZIAŁU 8:
--- Fragment ---
8. ALARM LIST
464
8.1 Alarms Displayed on NC Screen
8.1.1 Program errors (P/S alarm)
No. Message Contents
000 PLEASE TURN OFF
POWER
A parameter which requires the power off
was input, turn off power.
001 TH PARITY ALARM TH alarm (A character with incorrect parity
was input). Correct the program or tape.
002 TV PARITY ALARM TV alarm (The number of characters in a
block is odd). This alarm will be gen

In [ ]:
try:
    response = ollama.generate(model='llama3', prompt='Cześć, czy mnie słyszysz?')
    print("Połączenie z Ollama: DZIAŁA!")
    print(f"Odpowiedź: {response['response']}")
except Exception as e:
    print(f"Błąd połączenia: {e}. Sprawdź, czy aplikacja Ollama jest włączona!")

Połączenie z Ollama: DZIAŁA!
Odpowiedź: Cześć! Tak, mnie słychać. Jak się masz?


In [20]:
def generate_final_answer(query, n_chunks=5):
    # 1. Połączenie z bazą i modelem embeddingowym
    client = chromadb.PersistentClient(path="robodrill_db")
    collection = client.get_collection("robodrill")
    model_emb = load_embedding_model() # Używamy Twojej funkcji do załadowania modelu

    # 2. RETRIEVAL: Szukamy 5 najlepszych fragmentów
    print(f"-> Przeszukuję instrukcję dla hasła: '{query}'...")
    query_emb = model_emb.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=n_chunks)
    
    # Składamy kontekst i zapamiętujemy rozdziały dla transparentności
    context_parts = []
    sources = set()
    for i in range(len(results['documents'][0])):
        doc_text = results['documents'][0][i]
        chapter = results['metadatas'][0][i]['chapter']
        context_parts.append(f"[ROZDZIAŁ: {chapter}]\n{doc_text}")
        sources.add(chapter)
    
    full_context = "\n\n".join(context_parts)

    # 3. PROMPT: Instrukcja dla LLM
    # Ustawiamy go tak, aby był konkretnym serwisantem
    prompt = f"""
    Jesteś ekspertem serwisu maszyn Fanuc Robodrill. 
    Twoim zadaniem jest pomóc technikowi naprawić maszynę na podstawie poniższych fragmentów instrukcji.

    ZASADY:
    1. Odpowiedz konkretnie i technicznie.
    2. Jeśli fragmenty zawierają tabelę lub listę kroków, przedstaw je czytelnie.
    3. Jeśli nie znajdziesz odpowiedzi w tekście, napisz uczciwie, że instrukcja nie podaje rozwiązania dla tego błędu.
    4. Odpowiedz w języku polskim, ale zachowaj oryginalne nazwy techniczne i kody błędów.

    FRAGMENTY Z INSTRUKCJI:
    {full_context}

    PYTANIE TECHNIKA: 
    {query}

    ODPOWIEDŹ EKSPERTA:
    """

    # 4. GENERATION: Wywołanie Llamy
    print(f"-> Model AI generuje odpowiedź na podstawie {len(sources)} różnych sekcji...")
    response = ollama.generate(model='llama3', prompt=prompt)

    # 5. Wyświetlenie wyniku
    print("\n" + "="*60)
    print("🤖 ASYSTENT SERWISU FANUC ROBODRILL")
    print("="*60)
    print(response['response'])
    print("\n" + "-"*60)
    print(f"📚 Źródła (rozdziały): {', '.join(sources)}")
    print("="*60)

In [21]:
# URUCHOMIENIE TESTU:
generate_final_answer("Error code AL-24 description")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Przeszukuję instrukcję dla hasła: 'Error code AL-24 description'...
-> Model AI generuje odpowiedź na podstawie 1 różnych sekcji...

🤖 ASYSTENT SERWISU FANUC ROBODRILL
W oparciu o fragmenty instrukcji, kod błędu AL-24 nie jest wymieniony. Niestety, w tym przypadku nie mogę udzielić szczegółowej odpowiedzi na pytanie o error code AL-24, ponieważ instrukcja nie zawiera tej informacji.

Jeśli chcesz uzyskać więcej informacji o błędzie AL-24, proszę o podanie dalszych fragmentów instrukcji lub kontakt z producentem maszyny Fanuc Robodrill.

------------------------------------------------------------
📚 Źródła (rozdziały): 8. ALARM LIST
